Confirmar o ambiente e ler a camada Bronze

In [0]:
#Confirmar que o Spark está ativo
print(f"Spark version: {spark.version}")

Spark version: 4.1.0


In [0]:
#Ler direto da tabela Bronze registrada no catálogo
df_bronze = spark.table("workspace.lakehouse_edu.bronze_student_performance")

In [0]:
#Confirmar o que temos
print(f"\nTotal de registros na Bronze: {df_bronze.count()}")
print(f"Total de colunas: {len(df_bronze.columns)}")

Renomear colunas para nomes descritivos

In [0]:
from pyspark.sql import functions as F
#Renomear colunas para nomes descritivos em português
df_renamed = df_bronze.select(
    F.col("school").alias("escola"),
    F.col("sex").alias("sexo"),
    F.col("age").alias("idade"),
    F.col("address").alias("tipo_endereco"),
    F.col("famsize").alias("tamanho_familia"),
    F.col("Pstatus").alias("status_pais"),
    F.col("Medu").alias("escolaridade_mae"),
    F.col("Fedu").alias("escolaridade_pai"),
    F.col("Mjob").alias("profissao_mae"),
    F.col("Fjob").alias("profissao_pai"),
    F.col("reason").alias("motivo_escolha_escola"),
    F.col("guardian").alias("responsavel"),
    F.col("traveltime").alias("tempo_deslocamento"),
    F.col("studytime").alias("horas_estudo_semana"),
    F.col("failures").alias("reprovacoes_anteriores"),
    F.col("schoolsup").alias("suporte_escolar"),
    F.col("famsup").alias("suporte_familiar"),
    F.col("paid").alias("aulas_pagas"),
    F.col("activities").alias("atividades_extracurriculares"),
    F.col("nursery").alias("frequentou_creche"),
    F.col("higher").alias("quer_ensino_superior"),
    F.col("internet").alias("tem_internet"),
    F.col("romantic").alias("tem_relacionamento"),
    F.col("famrel").alias("qualidade_relacao_familiar"),
    F.col("freetime").alias("tempo_livre"),
    F.col("goout").alias("frequencia_saidas"),
    F.col("Dalc").alias("consumo_alcool_semana"),
    F.col("Walc").alias("consumo_alcool_fds"),
    F.col("health").alias("saude"),
    F.col("absences").alias("faltas"),
    F.col("G1").alias("nota_1_bimestre"),
    F.col("G2").alias("nota_2_bimestre"),
    F.col("G3").alias("nota_final"),
    F.col("_ingestion_timestamp"),
    F.col("_source_file")
)

In [0]:
#Confirmar novas colunas
print(f"Total de colunas após renomear: {len(df_renamed.columns)}")
display(df_renamed.limit(5))

Verificar nulos e duplicatas

In [0]:
#Verificar nulos por coluna
print("Contagem de nulos por coluna:")
df_renamed.select([
    F.count(F.when(F.col(c).isNull(),c)).alias(c)
    for c in df_renamed.columns
]).display()

escola,sexo,idade,tipo_endereco,tamanho_familia,status_pais,escolaridade_mae,escolaridade_pai,profissao_mae,profissao_pai,motivo_escolha_escola,responsavel,tempo_deslocamento,horas_estudo_semana,reprovacoes_anteriores,suporte_escolar,suporte_familiar,aulas_pagas,atividades_extracurriculares,frequentou_creche,quer_ensino_superior,tem_internet,tem_relacionamento,qualidade_relacao_familiar,tempo_livre,frequencia_saidas,consumo_alcool_semana,consumo_alcool_fds,saude,faltas,nota_1_bimestre,nota_2_bimestre,nota_final,_ingestion_timestamp,_source_file
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
#Verificar Duplicatas
total = df_renamed.count()
distinct = df_renamed.dropDuplicates().count()
print(f"Total de registros: {total}")
print(f"Registros únicos: {distinct}")
print(f"Duplicatas encontradas: {total - distinct}")


Total de registros: 395
Registros únicos: 395
Duplicatas encontradas: 0


Aplicar transformações e regras de negócio

In [0]:
from pyspark.sql.types import IntegerType

df_silver = (
    df_renamed

    # Padronizar campos categóricos
    .withColumn("tipo_endereco",
        F.when(F.col("tipo_endereco") == "U", "urbano")
        .otherwise("rural")
    )
    .withColumn("sexo",
        F.when(F.col("sexo") == "F", "feminino")
        .otherwise("masculino")
    )

    # Garantir tipos corretos — usando select para evitar duplicatas
    .withColumns({
        "idade": F.col("idade").cast(IntegerType()),
        "faltas": F.col("faltas").cast(IntegerType()),
        "nota_1_bimestre": F.col("nota_1_bimestre").cast(IntegerType()),
        "nota_2_bimestre": F.col("nota_2_bimestre").cast(IntegerType()),
        "nota_final": F.col("nota_final").cast(IntegerType())
    })

    # REGRA DE NEGÓCIO 1 — Status de aprovação
    .withColumn("status_aprovacao",
        F.when(F.col("nota_final") >= 10, "aprovado")
        .when(F.col("nota_final") == 0, "sem_avaliacao")
        .otherwise("reprovado")
    )

    # REGRA DE NEGÓCIO 2 — Média das três notas
    .withColumn("media_notas",
        F.round(
            (F.col("nota_1_bimestre") + F.col("nota_2_bimestre") + F.col("nota_final")) / 3,
            2
        )
    )

    # Metadados da camada Silver
    .withColumn("_silver_timestamp", F.current_timestamp())
    .withColumn("_layer", F.lit("silver"))
)

print(f"Total de registros na Silver: {df_silver.count()}")
print(f"Total de colunas na Silver: {len(df_silver.columns)}")
display(df_silver.limit(5))


Validação pós transformação

In [0]:
#Distribuição de status de aprovação
print("Distribuição de status de aprovação:")
df_silver.groupBy("status_aprovacao").count().display()

In [0]:
#Distribuição por escola
print("Distribuição por escola:")
df_silver.groupBy("escola").count().display()

In [0]:
#Distribuição por tipo de endereço
print("Distribuição por tipo de endereço:")
df_silver.groupBy("tipo_endereco").count().display()

In [0]:
#Distribuição por sexo
print("Distribuição por sexo:")
df_silver.groupBy("sexo").count().display()

Salvar como Delta Table na camada Silver

In [0]:
# Salvar e registrar diretamente como tabela gerenciada no catálogo
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.lakehouse_edu.silver_student_performance")
)
print("✅ Camada Silver salva com sucesso!")

✅ Camada Silver salva com sucesso!


In [0]:
#Validação final - ler direto no Catálogo
df_check = spark.table("workspace.lakehouse_edu.silver_student_performance")
print(f"Total de registros na Silver: {df_check.count()}")
print(f"Total de colunas na Silver: {len(df_check.columns)}")

Total de registros na Silver: 395
Total de colunas na Silver: 39
